In [34]:
import pandas as pd

In [35]:
df = pd.read_csv('../data/raw/binace_btc_1yr_1minGap_data.csv')
df.head()

,Open Time,Open Price,High Price,Low Price,Close Price,Volume
0,1742392860000,84195.05,84315.21,84195.04,84312.60,20.66571
1,1742392920000,84312.60,84373.83,84241.70,84287.93,60.00135
2,1742392980000,84287.92,84355.66,84280.01,84327.80,12.15415
3,1742393040000,84327.81,84372.99,84311.82,84365.38,21.61167
4,1742393100000,84365.38,84388.00,84322.49,84381.07,34.51707


In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 525606 entries, 0 to 525605
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Open Time    525606 non-null  int64  
 1   Open Price   525606 non-null  float64
 2   High Price   525606 non-null  float64
 3   Low Price    525606 non-null  float64
 4   Close Price  525606 non-null  float64
 5   Volume       525606 non-null  float64
dtypes: float64(5), int64(1)
memory usage: 24.1 MB


In [37]:
df.isnull().sum()

Open Time      0
Open Price     0
High Price     0
Low Price      0
Close Price    0
Volume         0
dtype: int64

In [38]:
# converting open time from unix millisecond to normal ms 
df['Open Time'] = pd.to_datetime(df['Open Time'] , unit="ms")
df.set_index('Open Time' , inplace=True)
df.head()

,Open Price,High Price,Low Price,Close Price,Volume
Open Time,,,,,
2025-03-19 14:01:00,84195.05,84315.21,84195.04,84312.60,20.66571
2025-03-19 14:02:00,84312.60,84373.83,84241.70,84287.93,60.00135
2025-03-19 14:03:00,84287.92,84355.66,84280.01,84327.80,12.15415
2025-03-19 14:04:00,84327.81,84372.99,84311.82,84365.38,21.61167
2025-03-19 14:05:00,84365.38,84388.00,84322.49,84381.07,34.51707


In [39]:
#missing 1 min frequency gaps 


#if theres a 1 min gap then pd will add a new row for it and 
#fill the row with na stuff 

df = df.resample('1min').asfreq() 


#forward fill will filll these na columns with the last filled value 

df[['Open Price' , 'High Price' , 'Low Price' , 'Close Price']] = df[['Open Price' , 'High Price' , 'Low Price' , 'Close Price']].ffill()

#fill vol with 0 
df['Volume'] = df['Volume'].fillna(0)


In [40]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 525606 entries, 2025-03-19 14:01:00 to 2026-03-19 14:06:00
Freq: min
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Open Price   525606 non-null  float64
 1   High Price   525606 non-null  float64
 2   Low Price    525606 non-null  float64
 3   Close Price  525606 non-null  float64
 4   Volume       525606 non-null  float64
dtypes: float64(5)
memory usage: 24.1 MB


In [41]:
# new features which will measure the percentage chnage over the last 1 min , 3 min and 5 min 

df['Change_1min'] = df['Close Price'].pct_change(periods=1)
df['Change_3min'] = df['Close Price'].pct_change(periods=3)
df['Change_5min'] = df['Close Price'].pct_change(periods=5)



In [42]:
# past 15 min normal vol 

df['Vol_avg_15min'] = df['Volume'].rolling(window=15).mean()

#volume spike ration 

df['Volume_spike_ratio'] = df['Volume']/df['Vol_avg_15min']

#high low apread 

df['High_Low_spread'] = (df['High Price'] - df['Low Price'])/df['Low Price']

In [43]:
import ta 

# this tells that the thing is normal , oversold , or overbought 

df['Relative_strength_index'] = ta.momentum.RSIIndicator(close=df['Close Price'] , window=14).rsi()

In [58]:
#target var

df['Target_Pump'] = (
    (df['Change_5min'] > 0.005) &
    (df['Volume_spike_ratio'] > 3.0)
).astype(int)

#its pump time when the change in last 5 min is greater than 2%
#and the volume has spiked 3 times

In [59]:
df['Target_Pump'].value_counts()

Target_Pump
0    525158
1       434
Name: count, dtype: int64

In [60]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 525592 entries, 2025-03-19 14:15:00 to 2026-03-19 14:06:00
Freq: min
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Open Price               525592 non-null  float64
 1   High Price               525592 non-null  float64
 2   Low Price                525592 non-null  float64
 3   Close Price              525592 non-null  float64
 4   Volume                   525592 non-null  float64
 5   Change_1min              525592 non-null  float64
 6   Change_3min              525592 non-null  float64
 7   Change_5min              525592 non-null  float64
 8   Vol_avg_15min            525592 non-null  float64
 9   Volume_spike_ratio       525592 non-null  float64
 10  High_Low_spread          525592 non-null  float64
 11  Relative_strength_index  525592 non-null  float64
 12  Target_Pump              525592 non-null  int64  
dtypes: float64(12), int64(1)
m

In [61]:
df.isnull().sum()

Open Price                 0
High Price                 0
Low Price                  0
Close Price                0
Volume                     0
Change_1min                0
Change_3min                0
Change_5min                0
Vol_avg_15min              0
Volume_spike_ratio         0
High_Low_spread            0
Relative_strength_index    0
Target_Pump                0
dtype: int64

In [62]:
df.dropna(inplace=True)

In [63]:
df.isnull().sum()

Open Price                 0
High Price                 0
Low Price                  0
Close Price                0
Volume                     0
Change_1min                0
Change_3min                0
Change_5min                0
Vol_avg_15min              0
Volume_spike_ratio         0
High_Low_spread            0
Relative_strength_index    0
Target_Pump                0
dtype: int64

In [64]:
import os

file_path = "../data/processed"
file_name = "processed_data.csv"
final_path = os.path.join(file_path , file_name)
df.to_csv(final_path)
